# CNN и аугментации

Авторы BelOAI. CC BY 4.0.

Выполняйте ячейки сверху вниз. Сначала решите задачи, затем обратитесь к решениям.

## Чему научимся

Собирать небольшую CNN и выбирать преобразования изображений, сохраняющие смысл метки.

## От локального фильтра к ответу

CNN соединяет свёртки, нелинейности и объединение информации. Несколько выходных каналов означают несколько разных фильтров. Pooling уменьшает пространственные размеры, например усредняя участки; после него классификатор может использовать компактное описание картинки.

**Аугментация** создаёт изменённые варианты train-изображений, чтобы сеть не запоминала случайную позицию или яркость. Преобразование допустимо, если правильная метка сохраняется или корректно изменяется вместе с данными. Зеркальное отражение обычного предмета может быть допустимо, но отражение буквы или дорожного знака может изменить смысл.

## Разобранный пример: горизонтальные полосы

Синтетическая картинка содержит яркую горизонтальную полосу. Сдвиг по вертикали сохраняет класс «горизонтальная», поворот на 90° превращает её в вертикальную и меняет класс. Поэтому нельзя назвать произвольный поворот безопасным увеличением данных. Для детекции и сегментации вместе с картинкой нужно преобразовывать рамки или маски.

В ноутбуке torch.flip отражает указанную ось, torch.rot90 поворачивает на четверть оборота. Мы проверяем эти преобразования на собственной картинке и форму выхода маленькой CNN. nn.AdaptiveAvgPool2d((1,1)) усредняет каждый канал до одной ячейки, nn.Flatten соединяет оставшиеся признаки после оси пакета, Linear выдаёт два счёта классов.

Validation обычно преобразуют детерминированно по заранее выбранному правилу. Если добавлять случайные аугментации при оценке, нужно отдельно описать процедуру усреднения и её стоимость; это уже другая модель применения.

## Разобранный пример: проверяем вычисление

Все числа в примере придуманы для этого урока. Выполните код, затем измените вход и объясните новый результат.

In [ ]:
import torch
from torch import nn
torch.set_num_threads(1)
torch.manual_seed(17)
image = torch.zeros(1, 1, 8, 8)
image[:, :, 3, :] = 1
reflected = torch.flip(image, dims=[3])
rotated = torch.rot90(image, k=1, dims=[2, 3])
assert torch.equal(reflected, image)
assert not torch.equal(rotated, image)
model = nn.Sequential(nn.Conv2d(1, 3, 3, padding=1), nn.ReLU(), nn.AdaptiveAvgPool2d((1, 1)), nn.Flatten(), nn.Linear(3, 2))
assert model(image).shape == (1, 2)
print(model(image).detach())


## Типичные ошибки

Не разделяйте варианты одного исходника между частями. Не меняйте изображение без связанных координатных ответов.

## Задачи

### 1. Базовая

Что означает выходной канал свёртки?

<details>
<summary>Подсказка</summary>

Каждый имеет свои веса фильтра.

</details>



### 2. Базовая

Все ли повороты безопасны для метки?

<details>
<summary>Подсказка</summary>

Смысл может измениться.

</details>



### 3. Прикладная

Горизонтальную полосу повернули на 90°. Сохранилась ли метка направления?

<details>
<summary>Подсказка</summary>

Она стала вертикальной.

</details>



### 4. Прикладная

Форма после pooling (4,3,1,1). Сколько признаков у объекта после Flatten?

<details>
<summary>Подсказка</summary>

Ось объектов не соединяется.

</details>



### 5. Олимпиадная

Почему две аугментации одного исходного снимка нельзя разделять между train и test?

<details>
<summary>Подсказка</summary>

Они почти один объект.

</details>



### 6. Олимпиадная

Почему при отражении картинки сегментации нужно отразить маску?

<details>
<summary>Подсказка</summary>

Ответ привязан к координатам пикселей.

</details>



## Итоги и самопроверка

Назовите безопасное и опасное преобразование для своей задачи. Проследите NCHW через все слои CNN.

Для перехода дальше достаточно базовых и прикладных задач. Олимпиадные задачи можно отложить.


## Решения задач

### Задача 1

Один из вычисленных локальных признаков.

### Задача 2

Нет. Допустимость определяется задачей.

### Задача 3

Нет, нужно изменить метку либо отказаться от такой аугментации.

### Задача 4

Три; итоговая форма (4,3).

### Задача 5

Проверка получит зависимую копию знакомого изображения и станет слишком лёгкой.

### Задача 6

Иначе признаки и правильные ответы будут описывать разные положения объектов.